# 04 · 進階:權力算子(9 個操作 × 4 種權力)
主冊「只調高度」只放開一個自由度。但真實的權力改的是**形態語法**:開發商把板樓拆成細高塔、
居民把大院細分成自建小屋、政府把高度向權力重心收攏。這本把這些「操作」教給你。

> 數據、角色、讀法都和主冊一樣;變的是——**權力現在可以改幾何,不只是高度**。

## 怎麼執行
- 點一格,按 **Shift+Enter** 執行;或選單 Run → Run All Cells 從頭跑。
- 結果與圖會顯示在該格下面。代碼都在 `engine/` 文件夾,平時不用打開。

In [ ]:
# 讓 notebook 找到 engine 裏的代碼(這格不用改,直接執行)
import sys, os
_R = os.path.abspath("." if os.path.exists("engine") else "..")   # 包根(notebook 在 教學用notebook/ 子夹也找得到)
for _p in (_R, os.path.join(_R, "engine")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
# 清掉舊的引擎模塊緩存:改了 config/engine 後,重跑本格即刻生效(免重啟內核)
for _m in [k for k in list(sys.modules) if k in ("config", "common", "operators", "measure")
           or k == "plots" or k.startswith("plots.") or k == "prints" or k.startswith("prints.")]:
    del sys.modules[_m]
import config, common as C, plots, prints
prints.ready()
plots.capture("04")   # 開啟自動存圖:本冊每張圖都會存到 out/<slug>/Step_04/<時間戳>/

In [ ]:
import operators as ops, measure as M
df = C.assign_all(C.current_buildings(config.SLUG))
recs = C.to_recs(df)            # 算子的工作單位:[{geom, h, sh, frozen}]
prints.op_list(recs)

## 一、9 個原子算子 = power 對 form 的"操作"
| 算子 | 做什麼 | 誰常用 |
|---|---|---|
| `freeze` | 鎖定不動 | state / 遺產 |
| `weight_height` | 按權重重分配高度 | 政府集權 |
| `concentrate` | 高度向權力重心收攏 | 政府集權(單峯) |
| `split_to_towers` | 大板樓拆成 k 座塔 | 開發商 |
| `slim` | footprint 縮、高度補償(塔化) | 開發商 |
| `densify` | 抬高度 = 加 GFA | 開發商 |
| `infill` | 大體量細分成自建小單元 | 居民自建 |
| `level` | 高度趨同 | 居民 / 共享 |
| `open_ground` | 縮私有 footprint 釋放共享地面 | 共享平權 |

每個都是純函數 `recs -> recs`,參數可改、可單獨演示。下面看兩個。

## 二、算子圖譜:單個操作的 before / after
**拆板成塔** `split_to_towers`:把大 footprint 沿長軸拆成 3 塊(高度不變 → GFA 守恆)。

In [ ]:
b = ops.freeze(recs, who=["state"])                                   # 先鎖定政府公共
a = ops.split_to_towers(b, target=["resident", "developer"], above_m2=1500, k=3)
plots.operator_demo(b, a, "拆板成塔 split_to_towers(k=3)")

**居民自建** `infill`:把大體量細分成自建小單元(細粒、低層、有機)。看棟數怎麼暴漲。

In [ ]:
a2 = ops.infill(b, target=["resident", "developer"], cell_m2=120, min_h=6, max_h=21)
plots.operator_demo(b, a2, "居民自建 infill(cell≈120㎡)", color="h")
ac = ops.concentrate(b)
plots.operator_demo(b, ac, "中央集權 concentrate(向權力重心收攏)", color="h")

## 三、權力體制 = 算子的配方(`regimes.yaml`)
把幾個操作**按序串起來**,就是一種權力如何重寫城市。4 種體制各有一張「配方」:

In [ ]:
regs = ops.load_regimes()
prints.regimes(regs)

## 四、四種權力 → 四種形態(同色階橫比)
同一座城市、同一批樓,套四種權力體制。**形態差異不是換了數據,純粹來自「誰更有權、用什麼操作」。**

In [ ]:
after = {n: ops.apply_regime(recs, regs[n]) for n in regs}
labels = {n: regs[n]["label"] for n in regs}
plots.regime_compare(recs, after, labels=labels)

## 五、形態特徵(measure.py 量化)
肉眼之外,用指標坐實每種權力的「特徵」:開發商=瘦長比飆升(細高塔);政府=重心集中度翻倍(向權力重心收攏);
居民=棟數暴漲(細粒碎化);共享=高度CV 最低(均質)。

In [ ]:
rows, _ = M.compare(recs, after, config.SLUG)
plots.feature_bars(rows, labels={"current": "現狀", **labels})
prints.features(rows, {"current": "現狀", **labels})

## 六、權力的幾何邏輯對基底不變,但結果在乎基底
把 `config.py` 的 `SLUG` 從 `lujiazui`(CBD 超高層)換成 `yuyuan`(里弄)或 `caoyang`(工人新村),
**重跑這本**:同一個 `developer_led` 配方,蓋在里弄和蓋在 CBD **不會長成一樣**。
> 權力 × 在地是一場對話,不是蓋章。這正是爲什麼我們堅持跑**真實多源數據**,而不是合成方塊。

## 七、動手:複製粘貼換一個算子
算子就是 `recs -> recs` 的純函數。下面**在 notebook 裏**定義一個你自己的算子、登記、用進配方——
不必改 `engine/`。完整模板與說明見 **`算子替換指南.md`**。

In [ ]:
import shapely.affinity as aff

def twist(recs, target, deg=18):
    """我的算子:把目標樓的 footprint 旋轉 deg 度(示範——你可以換成任何形態操作)。"""
    out = [dict(r) for r in recs]
    for r in out:
        if r["sh"] in target and not r.get("frozen"):
            r["geom"] = aff.rotate(r["geom"], deg, origin="centroid")
    return out

ops.register("twist", twist)                       # 登記到算子表(執行時,不改文件)
a3 = twist(b, target=["resident", "developer"], deg=25)
plots.operator_demo(b, a3, "我的算子 twist(旋轉 25°)")
prints.registered("twist")

**用進配方**:把 `twist` 寫進一條 `steps`,`apply_regime` 就會按序施加它——和內置算子完全平權。
```yaml
my_regime:
  steps:
    - {op: freeze, who: [state]}
    - {op: twist, target: [resident, developer], deg: 25}
    - {op: densify, target: [resident, developer], far_gain: 1.5}
```
> 這就是這套框架的核心承諾:**權力體制 = 算子的配方,而算子可教、可改、可複製粘貼。**

## 誠實邊界
這些 regime → 形態是**教學假設 / 可爭論的語言**,不是經驗斷言或真實規劃預測。
簡化算子不含退線/日照/產權/參與;`informal` 無信號恆空;`danwei` 國家屬性在用途數據裏看不見。零 AI 推斷。